# Feature Stores & Data Infrastructure — Azure ML Feature Store

**Chapter 8: AI Infrastructure (Azure Supplement)**

This notebook demonstrates production-grade feature store deployment using **Azure ML Feature Store** with Redis as the online store.

**Production Architecture:**
- **Offline store**: Azure Data Lake Storage (ADLS Gen2) with Parquet files
- **Online store**: Azure Cache for Redis (Enterprise tier, 99.99% SLA)
- **Feature serving**: Azure Functions (serverless, auto-scaling)
- **Monitoring**: Application Insights (latency, drift, staleness)

**Tools:**
- **Azure ML SDK**: Feature store management
- **Azure Cache for Redis**: Managed Redis (6GB-120GB)
- **Azure Functions**: Serverless feature serving
- **Application Insights**: Telemetry and monitoring

**Cost targets:**
- Redis: $120-$600/month (depending on tier)
- Azure Functions: $0.20/million requests
- ADLS: $0.02/GB/month

---

## Cell 1: Azure Credentials + ML Workspace Setup

**Required Azure resources:**
1. Azure ML Workspace
2. Azure Cache for Redis (Standard/Premium tier)
3. Azure Data Lake Storage Gen2
4. Service Principal with Contributor role

**Security note:** This cell contains credential placeholders. In production:
- Use Azure Key Vault for secrets
- Use Managed Identity for Azure resources
- Never commit credentials to Git

In [ ]:
# TODO: Implement this cell


## Cell 2: Upload Training Data to Azure Data Lake

Upload user activity data to ADLS Gen2 (offline store).

**Storage layout:**
```
features/
├── user_uploads/
│   └── year=2024/month=04/day=26/
│       └── user_uploads.parquet
├── user_activity_features/
└── user_doc_affinity/
```

**Cost:** $0.02/GB/month (ADLS Standard tier)

In [ ]:
# TODO: Implement this cell


## Cell 3: Define Azure ML Feature Set

Create feature set definitions using Azure ML SDK.

**Azure ML Feature Set** = Feast Feature View equivalent
- Source: ADLS Parquet files
- Entities: user_id
- Features: Aggregations over user activity

**Note:** Azure ML Feature Store uses declarative YAML + Python SDK (vs Feast's pure Python).

In [ ]:
# TODO: Implement this cell


## Cell 4: Register Feature Set to Azure ML

Register the feature set with Azure ML Feature Store.

**What this does:**
1. Validates feature definitions
2. Registers metadata to Azure ML workspace
3. Creates lineage tracking (source → features)
4. Enables feature discovery in Azure ML Studio UI

**Portal URL:** https://ml.azure.com → Feature stores

In [ ]:
# TODO: Implement this cell


## Cell 5: Materialize Features to Azure Redis

Compute aggregated features and write to Azure Cache for Redis (online store).

**Materialization job:**
1. Read user_uploads from ADLS
2. Compute aggregations (AVG, SUM, COUNT)
3. Write to Redis with TTL=30 days

**Redis key structure:**
```
fs:user_activity_features:user:12345:avg_confidence_score → 0.87
fs:user_activity_features:user:12345:total_pages_processed → 1250
```

**Cost:** Azure Cache for Redis Standard C1 (1GB) = $75/month

In [ ]:
# TODO: Implement this cell


## Cell 6: Online Serving with Azure Functions

Deploy serverless feature serving API using Azure Functions.

**Architecture:**
```
Client → Azure Functions (HTTP trigger)
           ↓
        Azure Cache for Redis (feature lookup)
           ↓
        Return features (JSON)
```

**Cost:** $0.20/million requests + $0.000016/GB-s execution time

**Deployment:** Via Azure Functions Core Tools or Azure Portal

In [ ]:
# TODO: Implement this cell


## Cell 7: Cost Analysis — Redis Tiers

Compare Azure Cache for Redis pricing tiers.

**Tiers:**
- **Basic**: No SLA, single node (dev/test only)
- **Standard**: 99.9% SLA, replication, no persistence
- **Premium**: 99.95% SLA, persistence, clustering, VNet
- **Enterprise**: 99.99% SLA, Redis modules (RediSearch, RedisJSON)

**Recommendation:** Standard C1 (1GB) for <10k users, Premium P1 (6GB) for production

In [ ]:
# TODO: Implement this cell


## Cell 8: Monitor Feature Serving with Application Insights

Track feature serving metrics using Azure Application Insights.

**Metrics to monitor:**
1. **Latency**: p50, p95, p99 of feature lookup time
2. **Cache hit rate**: % of requests served from Redis (vs fallback to DB)
3. **Feature staleness**: Time since last materialization
4. **Error rate**: % of failed feature lookups

**Alerts:**
- p95 latency > 20ms → scale Redis tier
- Cache hit rate < 95% → increase TTL or materialization frequency
- Feature staleness > 2 hours → materialization job failed

In [ ]:
# TODO: Implement this cell
